# Implementing an Animal Expert System

An example from [AI for Beginners Curriculum](http://github.com/microsoft/ai-for-beginners).

In this sample, we will implement a simple knowledge-based system to determine an animal based on some physical characteristics. The system can be represented by the following AND-OR tree (this is a part of the whole tree, we can easily add some more rules):

![](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/2-Symbolic/images/AND-OR-Tree.png?raw=1)

## Our own expert systems shell with backward inference

Let's try to define a simple language for knowledge representation based on production rules. We will use Python classes as keywords to define rules. There would be essentially 3 types of classes:
* `Ask` represents a question that needs to be asked to the user. It contains the set of possible answers.
* `If` represents a rule, and it is just a syntactic sugar to store the content of the rule
* `AND`/`OR` are classes to represent AND/OR branches of the tree. They just store the list of arguments inside. To simplify code, all functionality is defined in the parent class `Content`

In [1]:
class Ask():
    def __init__(self,choices=['y','n']):
        self.choices = choices
    def ask(self):
        if max([len(x) for x in self.choices])>1:
            for i,x in enumerate(self.choices):
                print("{0}. {1}".format(i,x),flush=True)
            x = int(input())
            return self.choices[x]
        else:
            print("/".join(self.choices),flush=True)
            return input()

class Content():
    def __init__(self,x):
        self.x=x

class If(Content):
    pass

class AND(Content):
    pass

class OR(Content):
    pass

In our system, working memory would contain the list of **facts** as **attribute-value pairs**. The knowledgebase can be defined as one big dictionary that maps actions (new facts that should be inserted into working memory) to conditions, expressed as AND-OR expressions. Also, some facts can be `Ask`-ed.

In [8]:
Rule = {
    'default': Ask(['y','n']),
    #rationalism
    'concept' : Ask(['socratic_method','theory_of_forms','cogito_ergo_sum',
                     'primary_secondary_qualities', 'problem_of_induction', 'peripatetic_axiom', 'logical_atomism','brain_in_a_vat']),

    'school:rationalist': If(OR(['innate_ideas','a_priori','recollection','pure_reason'])),
    'school:empiricist': If(OR(['tabula_rasa','sensory_experience','impressions','logical_analysis'])),
    'ungulate': If(['mammal',OR(['has hooves','chews cud'])]),
    'bird': If(OR(['feathers',AND(['flies','lies eggs'])])),
    'animal:monkey' : If(['mammal','carnivor','color:red-brown','pattern:dark spots']),
    'animal:tiger' : If(['mammal','carnivor','color:red-brown','pattern:dark stripes']),
    'animal:giraffe' : If(['ungulate','long neck','long legs','pattern:dark spots']),
    'animal:zebra' : If(['ungulate','pattern:dark stripes']),
    'animal:ostrich' : If(['bird','long nech','color:black and white','cannot fly']),
    'animal:pinguin' : If(['bird','swims','color:black and white','cannot fly']),
    'animal:albatross' : If(['bird','flies well'])
}

To perform the backward inference, we will define `Knowledgebase` class. It will contain:
* Working `memory` - a dictionary that maps attributes to values
* Knowledgebase `rules` in the format as defined above

Two main methods are:
* `get` to obtain the value of an attribute, performing inference if necessary. For example, `get('color')` would get the value of a color slot (it will ask if necessary, and store the value for later usage in the working memory). If we ask `get('color:blue')`, it will ask for a color, and then return `y`/`n` value depending on the color.
* `eval` performs the actual inference, i.e. traverses AND/OR tree, evaluates sub-goals, etc.

In [3]:
class KnowledgeBase():
    def __init__(self,rules):
        self.rules = rules
        self.memory = {}

    def get(self,name):
        if ':' in name:
            k,v = name.split(':')
            vv = self.get(k)
            return 'y' if v==vv else 'n'
        if name in self.memory.keys():
            return self.memory[name]
        for fld in self.rules.keys():
            if fld==name or fld.startswith(name+":"):
                # print(" + proving {}".format(fld))
                value = 'y' if fld==name else fld.split(':')[1]
                res = self.eval(self.rules[fld],field=name)
                if res!='y' and res!='n' and value=='y':
                    self.memory[name] = res
                    return res
                if res=='y':
                    self.memory[name] = value
                    return value
        # field is not found, using default
        res = self.eval(self.rules['default'],field=name)
        self.memory[name]=res
        return res

    def eval(self,expr,field=None):
        # print(" + eval {}".format(expr))
        if isinstance(expr,Ask):
            print(field)
            return expr.ask()
        elif isinstance(expr,If):
            return self.eval(expr.x)
        elif isinstance(expr,AND) or isinstance(expr,list):
            expr = expr.x if isinstance(expr,AND) else expr
            for x in expr:
                if self.eval(x)=='n':
                    return 'n'
            return 'y'
        elif isinstance(expr,OR):
            for x in expr.x:
                if self.eval(x)=='y':
                    return 'y'
            return 'n'
        elif isinstance(expr,str):
            return self.get(expr)
        else:
            print("Unknown expr: {}".format(expr))

Now let's define our animal knowledgebase and perform the consultation. Note that this call will ask you questions. You can answer by typing `y`/`n` for yes-no questions, or by specifying number (0..N) for questions with longer multiple-choice answers.

In [4]:
kb = KnowledgeBase(rules)
kb.get('animal')

hair
y/n
n
gives milk
y/n
n
mammal
y/n
y
sharp teeth
y/n
y
claws
y/n
y
forward-looking eyes
y/n
y
color
0. red-brown
1. black and white
2. other
0
pattern
0. dark stripes
1. dark spots
0


'tiger'

## Using Experta for Forward Inference

In the next example, we will try to implement forward inference using one of the libraries for knowledge representation, [Experta](https://github.com/nilp0inter/experta). **Experta** is a library for creating forward inference systems in Python, which is designed to be similar to classical old system [CLIPS](http://www.clipsrules.net/index.html).

We could have also implemented forward chaining ourselves without many problems, but naive implementations are usually not very efficient. For more effective rule matching a special algorithm [Rete](https://en.wikipedia.org/wiki/Rete_algorithm) is used.

In [ ]:
import sys
!{sys.executable} -m pip install git+https://github.com/nilp0inter/experta

  Cloning https://github.com/nilp0inter/experta to /tmp/pip-req-build-7qurtwk3
  Running command git clone --filter=blob:none --quiet https://github.com/nilp0inter/experta /tmp/pip-req-build-7qurtwk3
  Resolved https://github.com/nilp0inter/experta to commit c6d5834b123861f5ae09e7d07027dc98bec58741
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for experta: filename=experta-1.9.5.dev1-py3-none-any.whl size=34804 sha256=888c459512a5e713f4b674caa9a0f96cfdf07ec0d6eb56cc318ce0653d218014
  Stored in directory: /tmp/pip-ephem-wheel-cache-1eeii9zy/wheels/3d/e8/bb/22d7956359603fa8dd679aa09f5b8efb3f29991c3986fdc787
Successfully built experta
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [experta]


In [ ]:
from experta import *
#import experta

We will define our system as a class that subclasses `KnowledgeEngine`. Each rule is defined by a separate function with `@Rule` annotation, which specifies when the rule should fire. Inside the rule, we can add new facts using `declare` function, and adding those facts will result in some more rules being called by forward inference engine.

In [10]:
class Philosopher(KnowledgeBase):
    @Rule(OR(
           (Fact(innate_ideas=True),
                  Fact(aPriori=True),
                  Fact(recollection=True),
                  Fact(pure_reason=True))))
    def id_rationalist(self):
        self.declare(Fact(school='Rationalist'))

    @Rule(OR(Fact(tabula_rasa=True),
             Fact(sensory_experience=True),
             Fact(impressions=True),
             Fact(logical_analysis=true)))
    def id_empiricist(self):
        self.declare(Fact(school='empiricist'))


    @Rule(Fact(school='Rationalist'),
          Fact(concept='socratic_method'))
    def socrates(self):
        self.declare(Fact(philosopher='Socrates', tradition='Rationalist(Proto Rationalism)'))


    @Rule(Fact(school='Rationalist'),
          Fact(concept='theory_of_forms'))
    def plato(self):
        self.declare(Fact(philosopher='Plato', tradition='Rationalist'))


    @Rule(Fact(school='Rationalist'),
          Fact(concept='cogito_ergo_sum'))
    def descartes(self):
        self.declare(Fact(philosopher='Descartes', tradition='Rationalist'))


    @Rule(Fact(school='Empiricist'),
          Fact(concept='primary_secondary_qualities'))
    def locke(self):
        self.declare(Fact(philosopher='Locke', tradition='Empiricist'))

    @Rule(Fact(school='Empiricist'),
          Fact(concept='problem_of_induction'))
    def hume(self):
        self.declare(Fact(philosopher='Hume', tradition='Empiricist'))

    @Rule(Fact(school='Empiricist'),
          Fact(concept='peripatetic_axiom'))
    def aquinas(self):
        self.declare(Fact(philosopher='Aquinas', tradition='Empiricist(Aristotelian)'))

    @Rule(Fact(Philosopher=MATCH.p, tradition=MATCH.t))
    def print_result(self,p,t):
          print(f"Philosopher is: {p} | School of thought is: {t}")

    def factz(self,facts):
        for f in facts:
            self.declare(f)

NameError: name 'Fact' is not defined

Once we have defined a knowledgebase, we populate our working memory with some initial facts, and then call `run()` method to perform the inference. You can see as a result that new inferred facts are added to the working memory, including the final fact about the animal (if we set up all the initial facts correctly).

In [ ]:
ex1 = Animals()
ex1.reset()
ex1.factz([
    Fact(color='red-brown'),
    Fact(pattern='dark stripes'),
    Fact('sharp teeth'),
    Fact('claws'),
    Fact('forward looking eyes'),
    Fact('gives milk')])
ex1.run()
ex1.facts

Animal is tiger


FactList([(0, InitialFact()),
          (1, Fact(color='red-brown')),
          (2, Fact(pattern='dark stripes')),
          (3, Fact('sharp teeth')),
          (4, Fact('claws')),
          (5, Fact('forward looking eyes')),
          (6, Fact('gives milk')),
          (7, Fact('mammal')),
          (8, Fact('carnivor')),
          (9, Fact(animal='tiger'))])